# 02 · Datos, particiones y preprocesamiento

Solo etapa 2 de la fase 1. Reutiliza el entorno virtual, raíz persistente y `ArtifactStore` de la etapa 1. No carga pesos ni ejecuta modelos, entrenamiento o evaluación final. Los reservados reciben únicamente comprobaciones estructurales.

El informe local y los manifiestos están en `docs/project/evidence/stage2/`; una ejecución local no acredita una ejecución de este notebook en Colab. La consulta al publicador sobre licencia sigue pendiente; adquisición solicitada explícitamente por el usuario el 2026-10-28. No redistribuir los GeoTIFF.


## 1. Conectar la misma persistencia y reutilizar el entorno

Para transportar esta implementación a Colab: ejecutar `python scripts/export_stage2_bundle.py`, subir el ZIP a Drive y copiar su SHA-256. Usa un directorio de fuente nuevo para conservar la fuente de recuperación de la etapa 1. En una sesión nueva, reconstruir el mismo venv con `scripts/bootstrap_colab.py` del paquete verificado; no repetir el ensayo prepare/recover de la etapa 1. Si el venv ya existe, se reutiliza.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage2')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage2.zip')
EXPECTED_BUNDLE_SHA256 = ''  # Resultado de export_stage2_bundle.py.
assert EXPECTED_BUNDLE_SHA256, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
                 AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42')
def command(args):
    result = subprocess.run(args, cwd=REPO, env=child_env, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
    return result
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')])
# Extensión aditiva: conserva los pines del entorno de etapa 1.
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')])
command([str(PYTHON), '-m', 'pip', 'check'])


## 2. Contratos y decisiones fijadas antes de procesar

`configs/data.yaml` fija percentil 99 exacto de valores RGB válidos agrupados (histograma entero exacto (int16 real; uint16 admitido)), exclusivamente del train de cada régimen: 252 o 213 chips. Se divide por 10000, se recorta a [0, upper] y se divide por upper. El límite no depende de etiquetas. No se aprende ninguna estadística en val, test, Bolivia o Paraguay reservado.

RGB: bandas GeoTIFF 4/3/2 = B4/B3/B2. SAM: 1024², media [0.485,0.456,0.406], std [0.229,0.224,0.225]. DINOv3 satelital: media [0.430,0.411,0.296], std [0.213,0.156,0.143]; tamaño elegido 512² sin crop ni padding, cuadrícula 32² de parches de 16. Ambas ramas usan resize bilinear antialias. Las máscaras usan nearest. Las fuentes primarias fijadas se conservan y verifican. Los originales v1.1 son int16 con nodata=0 en imagen y sin nodata declarado en etiquetas; los -1 de etiquetas se tratan explícitamente. Se confirma el orden RGB también en las descripciones GeoTIFF. Mekong corresponde al evento Cambodia de los metadatos: se conserva el nombre original y se comprueba la pertenencia espacial. Alineación imagen/etiqueta: tolerancia prefijada 1e-6 píxeles para diferencias de redondeo.

La validez de imagen procede solo de masks/nodata y valores finitos de RGB. La validez de etiquetas (`label != -1`) se almacena por separado para futuras pérdidas/métricas; nunca entra en `prepare`. Los reservados no generan cachés de píxeles ni exploración.


In [ ]:
command([str(PYTHON), '-m', 'unittest', 'discover', '-s', str(REPO / 'tests'), '-v'])


## 3. Adquirir, auditar y persistir

La adquisición descarga solo S2Hand, LabelHand, cuatro CSV oficiales y metadatos de eventos. Verifica generación GCS, tamaño y MD5 oficial; calcula SHA-256. Se reanuda reutilizando únicamente archivos íntegros. La auditoría verifica todos los pares, IDs, eventos, dimensiones, dtype, CRS y alineación, clases -1/0/1, duplicados por archivo y contenido de imagen, y vecindad/solapamiento. No obtiene estadísticas de contenido de los reservados.

Resultados: `official` 252/89/90/15; `leave_paraguay_out` 213/75/76/67 y Bolivia 15 apartada. Todo se conserva en PERSIST_ROOT: raw, manifests y preprocessing. Se guardan transformaciones y máscaras de validez solo de train/val permitidos.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/prepare_stage2.py'),
         '--root', PERSIST_ROOT,
         '--evidence', str(Path(PERSIST_ROOT) / 'stage2/evidence')])


## 4. Recuperación en otro proceso y nueva sesión

El paso anterior compara bit a bit ambas ramas tras cargar los parámetros en un intérprete nuevo. Para comprobar también el almacenamiento de Drive tras desconectar, reiniciar Colab, repetir la conexión y ejecutar la celda siguiente: `--offline` exige originales y fuentes ya descargadas, verifica hashes y reutiliza los parámetros compatibles sin volver a aprenderlos. No se presenta la desconexión de Colab como ejecutada en la evidencia local.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/prepare_stage2.py'),
         '--root', PERSIST_ROOT, '--offline',
         '--evidence', str(Path(PERSIST_ROOT) / 'stage2/evidence')])
report = json.loads((Path(PERSIST_ROOT) / 'stage2/evidence/validation.json').read_text())
print(json.dumps({k: {'counts': v['counts'], 'upper_reflectance': v['upper_reflectance'],
                     'reload': v['reload'], 'parameters_reused': v['parameters_reused']}
                  for k, v in report['regimes'].items()}, indent=2))
assert report['models_executed'] is False
assert report['reserved_pixel_caches'] == []


## 5. Contrato para etapas siguientes y cierre

`development_records(manifest, regime=..., role='train'|'val')` exige régimen y rol. `load_preprocessing` exige hashes de manifiesto/config y comprueba IDs fuente. `prepare(..., model='sam'|'dino')` devuelve tensores CHW float32 ya normalizados para una futura entrada directa al modelo; se debe omitir cualquier processor posterior. Para `SAM2ImagePredictor.set_image`, usar exclusivamente `sam_predictor_image`: float32 HWC [0,1], con una sola normalización aplicada por el wrapper oficial. Nunca pasar el tensor normalizado a `set_image`.

Coordenadas SAM: (x,y) × (target_w/original_w,target_h/original_h). Para interpolación se documenta la convención de centros y para DINO las celdas y centros de parches, con inversa analítica. La geometría GeoTIFF conserva unidades de CRS: EPSG:4326 está en grados, no se interpreta su resolución como 10 metros exactos.

Detenerse aquí. Pendientes: respuesta del publicador y revisión humana del código/decisiones; conservar si se desea la evidencia de la nueva sesión Colab. Las etapas 3–9 no se ejecutan. Asistencia de IA: Codex implementó módulos, notebook, pruebas, auditoría y registro; sin subagentes, datos sintéticos presentados como reales, predicciones ni resultados de segmentación.
